# DP-ADAGE: Differentially Private Active Defenses Against GNN Extraction
### Google Colab Experiment & Replication Pipeline

This notebook reproduces the experimental results, evaluations, and figures for **DP-ADAGE** (Differentially Private Active Defenses Against GNN Extraction).

**Workflow Overview:**
1. **Environment Setup**: Clone repository, set workspace directory, verify files.
2. **Dependencies**: Install required packages (`torch`, `torch_geometric`, `python-louvain`, etc.).
3. **Main Benchmarks**: Run extraction defense evaluations on individual datasets (Cora) and across all 5 benchmark graphs.
4. **Advanced Evaluations**: Test against adaptive attackers, large-scale graphs (Flickr), and profile memory/throughput.
5. **Visualizations**: Generate publication-ready figures (Figures 2, 3, 7, and summary PDF plots).

## 1. Environment & Repository Setup
Clone the official DP-ADAGE repository and navigate into the project workspace directory.

In [ ]:
# ==============================================================================
# Step 1: Clone Repository
# Clone the DP-ADAGE project repository from GitHub to access source code,
# experiment scripts, default configurations, and dataset utilities.
# ==============================================================================
!git clone https://github.com/jboussouf/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction.git

Cloning into 'DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 69 (delta 12), reused 69 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 259.01 KiB | 6.32 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r "/content/drive/MyDrive/test/fl/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction" "/content/"

In [ ]:
# ==============================================================================
# Step 2: Set Working Directory
# Switch the current working directory to the cloned repository root folder.
# ==============================================================================
import os
os.chdir('/content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction')

In [ ]:
# ==============================================================================
# Step 3: Verify Workspace Structure
# List files and folders to ensure all scripts, configs, and modules are present.
# ==============================================================================
!ls -l

total 344
drwx------ 2 root root   4096 Oct 23 10:25 data
drwx------ 2 root root   4096 Oct 23 10:25 experiments
drwx------ 2 root root   4096 Oct 23 10:25 figures
-rw------- 1 root root  30031 Oct 23 10:25 legitimate_user_evaluation.pdf
-rw------- 1 root root 176422 Oct 23 10:25 main_figure.png
-rw------- 1 root root  29020 Oct 23 10:24 README.md
drwx------ 3 root root   4096 Oct 23 10:25 reports
-rw------- 1 root root    948 Oct 23 10:25 requirements.txt
drwx------ 3 root root   4096 Oct 23 10:25 results
-rw------- 1 root root  68281 Oct 23 10:25 run_DP_ADAGE.ipynb
drwx------ 3 root root   4096 Oct 23 10:25 src
drwx------ 3 root root   4096 Oct 23 10:25 tests
drwx------ 3 root root   4096 Oct 23 10:25 visuals


## 2. Install Project Dependencies
Install all required packages specified in `requirements.txt`.

In [ ]:
# ==============================================================================
# Step 4: Install Dependencies
# Installs PyTorch, PyTorch Geometric, NetworkX, python-louvain, scikit-learn,
# matplotlib, and other required scientific computing packages.
# ==============================================================================
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.3 MB/s eta 0:00:00


## Legitimate User Evaluation
Simulate and evaluate a **legitimate user** interacting with the DP-ADAGE protected GNN API over 200 sequential queries.

**What this script does:**
- Mocks the DP-ADAGE Hard Floor mechanism (budget exhaustion at query 20)
- Tracks rolling prediction accuracy (10-query window) and cumulative privacy budget
- Saves a dual-axis PDF figure: `legitimate_user_evaluation.pdf`
- Prints a phase-split summary report to the console

In [ ]:
# ==============================================================================
# Legitimate User Evaluation (200 Queries)
# Simulates a benign user querying a localized neighborhood (τ ≤ τ_dead = 0.05).
# Budget is exhausted after exactly 20 queries (ε_total=200 / ε_max=10).
# Post-exhaustion: API returns uniform random labels (Hard Floor) — no blocking.
# Output: console summary report + 'legitimate_user_evaluation.pdf'
# ==============================================================================
!python experiments/evaluate_legitimate_user.py

[*] Running legitimate user simulation …

────────────────────────────────────────────────────────────
  DP-ADAGE  –  Legitimate User Evaluation Summary
────────────────────────────────────────────────────────────
  Dataset (simulated) : Cora-like (7 classes)
  Total queries       : 200
  ε_total             : 200.0
  ε_max (dead-zone)   : 10.0
  τ_dead              : 0.05
  Budget exhaustion   : after query 20
────────────────────────────────────────────────────────────
  [Phase 1]  Queries  1 –  20  (accurate phase)
             Avg accuracy  : 100.0%  (expected ~100.0%)

  [Phase 2]  Queries  21 – 200  (hard floor phase)
             Avg accuracy  : 15.6%  (random baseline ≈ 14.3%)

  [Overall]  API Availability : 100.0%
             (DP-ADAGE never blocks; it degrades gracefully)
────────────────────────────────────────────────────────────


[✓] Figure saved → legitimate_user_evaluation.pdf

[✓] Evaluation complete.


## 3. Main Benchmark Experiments
Evaluate extraction defense mechanisms across victim GNN architectures and surrogate models.

In [ ]:
# ==============================================================================
# Step 5: Quick Benchmark on Cora Dataset
# Compares NoDefense, StaticNoise, OriginalADAGE, and DP-ADAGE on Cora (Table 1 & Figure 2).
# Measures surrogate extraction accuracy and downstream accuracy over 100 queries.
# ==============================================================================
!python experiments/exp_main_results.py

Running Main Experiment (Table 1 & Figure 2)...

--- Running with seed 42 ---
Processing...
Done!
Successfully loaded real Cora dataset.
Dataset stats: {'nodes': 2708, 'edges': 10556, 'features': 1433}
Training victim model (GCN)...
Setting up defenses...

Evaluating against NoDefense...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against StaticNoise...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0003

Evaluating against OriginalADAGE...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0001

Evaluating against PRADA...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against DAWN...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against DP-ADAGE...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0468

--- Running with seed 43 ---
Successfully loaded real Cora dataset.
Dataset stats: {'nodes': 2708, 'edges': 10556, 'features': 1433}
Training victim model (GCN)...
Setting up defenses...

E

In [ ]:
# ==============================================================================
# Step 6: Full Evaluation Across All 5 Benchmark Datasets (Table 1)
# Runs comprehensive evaluation across Cora, Citeseer, Pubmed, ACM, and DBLP.
# Parameters:
#   --epochs 200      : Number of training epochs for victim & surrogate models
#   --max_queries 200 : Maximum query budget allocated for the extraction attacker
# Output: Results saved to 'data/results_all_datasets.csv'
# ==============================================================================
!python experiments/run_all_datasets.py --epochs 200 --max_queries 200

Streaming output truncated to the last 5000 lines.
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0055
[PRADA] Attack detected for user atk__Pubmed__PRADA__eps1.0__seed42! (p_val=0.0002)
    [Debug] Surrogate Training - Queries: 110, Loss: 0.0126
    [Debug] Surrogate Training - Queries: 120, Loss: 0.0133
    [Debug] Surrogate Training - Queries: 130, Loss: 0.0146
    [Debug] Surrogate Training - Queries: 140, Loss: 0.0114
    [Debug] Surrogate Training - Queries: 150, Loss: 0.0155
    [Debug] Surrogate Training - Queries: 160, Loss: 0.0184
    [Debug] Surrogate Training - Queries: 170, Loss: 0.0185
    [Debug] Surrogate Training - Queries: 180, Loss: 0.0180
    [Debug] Surrogate Training - Queries: 190, Loss: 0.0182
    [Debug] Surrogate Training - Queries: 200, Loss: 0.0180
[Pubmed]           PRADA | Surrogate Acc: 0.6920 | Downstream_Acc: 0.7930 | Attacker Acc: 0.3510 | Budget ε: 0.0000 | Latency: 13309.6 ms

[Pubmed] Evaluating DAWN ...
[Pubmed]   Computing Downstream_Acc (

## 4. Advanced Evaluations: Adaptive Attackers, Large Graphs & Profiling
Stress-test DP-ADAGE against adaptive query strategies, large-scale graphs, and high-concurrency workloads.

In [ ]:
# ==============================================================================
# Step 7: Adaptive Attacker Evaluation
# Evaluates defenses against uncertainty-sampling and budget-aware adaptive attackers
# over 500 queries to assess the robustness of the hard RDP budget cap (epsilon_total).
# Output: Results saved to 'data/results_adaptive_attacker.csv'
# ==============================================================================
!python3 experiments/exp_adaptive_attacker.py

Running Adaptive Attacker Experiment (500 queries)...

--- Running with seed 42 ---
Successfully loaded real Cora dataset.
Training victim model (GCN)...
[INFO] Using GPU (CUDA) for evaluation.
Setting up defenses...

Evaluating against NoDefense with 'uncertainty' strategy...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0001
    [Debug] Surrogate Training - Queries: 110, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 120, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 130, Loss: 0.0001
    [Debug] Surrogate Training - Queries: 140, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 150, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 160, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 170, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 180, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 190, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 200, Loss: 0.0002
    [Debug] Surrogate Training - Queries: 210, Loss: 0.0002
  

In [ ]:
# ==============================================================================
# Step 8: Large-Scale Graph Scalability Evaluation (Flickr)
# Evaluates DP-ADAGE and NoDefense on the Flickr dataset (~89k nodes, ~450k edges).
# Measures offline Louvain partition time and per-query inference latency.
# Output: Results saved to 'data/results_large_scale.csv'
# ==============================================================================
!python3 experiments/exp_large_scale.py

Running Large-Scale Graph Evaluation (Flickr - 89k nodes)...
[INFO] Using GPU (CUDA) for evaluation.
Loading Flickr dataset... (This may take a moment to download)
Processing...
Done!
Loaded Flickr: 89250 nodes, 899756 edges, 500 features, 7 classes.

Training Victim Model (GCN)...

--- Evaluating NoDefense ---
[NoDefense] Preprocessing took: 0.00 seconds
[NoDefense] Running Attacker (100 queries)...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002
[NoDefense] Total attack time: 26.23 s | Avg Latency: 262.34 ms/query

--- Evaluating DP-ADAGE ---
[DP-ADAGE] Running preprocessing (e.g., Louvain)...
[DP-ADAGE] Preprocessing took: 32.46 seconds
[DP-ADAGE] Running Attacker (100 queries)...
    [Debug] Surrogate Training - Queries: 100, Loss: 1.5266
[DP-ADAGE] Total attack time: 26.52 s | Avg Latency: 265.23 ms/query

Results saved to data/results_large_scale.csv


In [ ]:
# ==============================================================================
# Step 9: Memory & Throughput Profiling
# Profiles DP-ADAGE under high concurrent user load (1000 simulated users).
# Measures memory footprint (CommunityTracker / RDPAccountant) and throughput (QPS).
# ==============================================================================
!python3 experiments/exp_profiling.py

In [ ]:
# ==============================================================================
# Step 10: Plot Main Results (Figure 2)
# Generates line plots of attacker surrogate accuracy and cumulative RDP budget
# consumption trajectory over queries.
# Output: Saved to 'visuals/outputs/fig2_main_results.pdf'
# ==============================================================================
!python visuals/plot_main_results.py

Plot saved to visuals/outputs/fig2_main_results.pdf


In [ ]:
# ==============================================================================
# Step 11: Plot Pareto Frontier (Figure 3)
# Plots the privacy vs. utility trade-off (Pareto frontier), with bubble size
# representing efficiency/latency across defense methods.
# Output: Saved to 'visuals/outputs/fig3_pareto.pdf'
# ==============================================================================
!python visuals/plot_pareto.py

In [ ]:
# ==============================================================================
# Step 12: Plot Radar Chart (Figure 7)
# Generates multidimensional radar chart comparing defense efficacy,
# utility preservation, latency overhead, and budget efficiency.
# Output: Saved to 'visuals/outputs/fig7_radar.pdf'
# ==============================================================================
!python visuals/plot_radar.py

Plot saved to visuals/outputs/fig7_radar.pdf


In [ ]:
# ==============================================================================
# Step 13: Generate All Publication Figures & Trajectories
# Parses all result CSV files and produces the full suite of figures (accuracy trajectories,
# budget depletion curves, ablation studies, and adaptive attacker response).
# Output: Saved to 'figures/'
# ==============================================================================
!python visuals/plot_all_results.py

[✓] Loaded main CSV: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../data/results_all_datasets.csv (3000 rows)
[✓] Loaded ablation CSV: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../data/results_ablation_budget.csv (1500 rows)
[✓] Loaded adaptive CSV: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../data/results_adaptive_attacker.csv (3000 rows)

Generating figures into: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../figures

[✓] Saved: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../figures/fig2_surrogate_trajectory.pdf
[✓] Saved: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../figures/fig3_budget_trajectory.pdf
[✓] Saved: /content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/visuals/../figures/fig4_abla

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
def aggregate_mean_std(input_file, output_file, group_cols, metric_cols):
    """
    Reads a CSV, groups by experimental conditions, and formats metrics as 'Mean ± Std'.
    """
    print(f"Processing {input_file}...")
    df = pd.read_csv(input_file)

    # Check if we actually have multiple runs per group
    group_counts = df.groupby(group_cols).size()
    max_runs = group_counts.max()

    if max_runs == 1:
        print(f"WARNING: Only 1 run found per configuration in {input_file}.")
        print("Standard deviation will be NaN. Ensure your logging script saves all 5 seeds!\n")
    else:
        print(f"Found {max_runs} runs per configuration. Calculating Mean ± Std...\n")

    # Group by the experimental conditions
    grouped = df.groupby(group_cols)

    # Calculate mean and std
    agg_mean = grouped[metric_cols].mean()
    agg_std = grouped[metric_cols].std()

    # Create a new DataFrame for the formatted results
    result_df = pd.DataFrame(index=agg_mean.index)

    for col in metric_cols:
        formatted_values = []
        for m, s in zip(agg_mean[col], agg_std[col]):
            if pd.isna(s) or s == 0:
                # If std is NaN (only 1 run) or 0, just print the mean
                formatted_values.append(f"{m:.3f}")
            else:
                # Format as Mean ± Std with 3 decimal places
                formatted_values.append(f"{m:.3f} ± {s:.3f}")
        result_df[col] = formatted_values

    # Reset index to flatten the DataFrame and save
    result_df = result_df.reset_index()
    result_df.to_csv(output_file, index=False)
    print(f"Saved aggregated results to {output_file}\n")
    return result_df

In [ ]:
# 1. For results_all_datasets.csv and results_main_cora.csv
main_group_cols = ['Dataset', 'Defense', 'Queries']
main_metric_cols = ['Surrogate_Acc', 'Surrogate_Fid', 'Downstream_Acc',
                    'Budget_Consumed', 'Tau', 'Latency_ms', 'KL_Divergence',
                    'Attacker_Downstream_Acc']

# 2. For results_ablation_budget.csv (includes Budget_Cap)
ablation_group_cols = ['Dataset', 'Defense', 'Queries', 'Budget_Cap']
ablation_metric_cols = ['Surrogate_Acc', 'Surrogate_Fid', 'Downstream_Acc',
                        'Budget_Consumed', 'Tau', 'Latency_ms', 'KL_Divergence',
                        'Attacker_Downstream_Acc']

In [ ]:
aggregate_mean_std(
    input_file='data/results_all_datasets.csv',
    output_file='data/results_all_datasets_aggregated.csv',
    group_cols=main_group_cols,
    metric_cols=main_metric_cols
)


Processing data/results_all_datasets.csv...
Found 5 runs per configuration. Calculating Mean ± Std...

Saved aggregated results to data/results_all_datasets_aggregated.csv



,Dataset,Defense,Queries,Surrogate_Acc,Surrogate_Fid,Downstream_Acc,Budget_Consumed,Tau,Latency_ms,KL_Divergence,Attacker_Downstream_Acc
0,ACM,DAWN,10,0.365 ± 0.067,0.385 ± 0.067,0.842 ± 0.020,0.000,0.000,264.026 ± 2.218,0.125 ± 0.119,0.809 ± 0.057
1,ACM,DAWN,20,0.548 ± 0.117,0.593 ± 0.114,0.842 ± 0.020,0.000,0.000,264.026 ± 2.218,0.125 ± 0.119,0.809 ± 0.057
2,ACM,DAWN,30,0.656 ± 0.099,0.723 ± 0.107,0.842 ± 0.020,0.000,0.000,264.026 ± 2.218,0.125 ± 0.119,0.809 ± 0.057
3,ACM,DAWN,40,0.707 ± 0.103,0.777 ± 0.109,0.842 ± 0.020,0.000,0.000,264.026 ± 2.218,0.125 ± 0.119,0.809 ± 0.057
4,ACM,DAWN,50,0.745 ± 0.113,0.821 ± 0.112,0.842 ± 0.020,0.000,0.000,264.026 ± 2.218,0.125 ± 0.119,0.809 ± 0.057
...,...,...,...,...,...,...,...,...,...,...,...
595,Pubmed,StaticNoise,160,0.787 ± 0.008,0.937 ± 0.006,0.793 ± 0.004,0.000,0.000,651.568 ± 6.184,0.043 ± 0.003,0.795 ± 0.003
596,Pubmed,StaticNoise,170,0.788 ± 0.007,0.939 ± 0.008,0.793 ± 0.004,0.000,0.000,651.568 ± 6.184,0.043 ± 0.003,0.795 ± 0.003
597,Pubmed,StaticNoise,180,0.789 ± 0.004,0.943 ± 0.005,0.793 ± 0.004,0.000,0.000,651.568 ± 6.184,0.043 ± 0.003,0.795 ± 0.003
598,Pubmed,StaticNoise,190,0.788 ± 0.003,0.942 ± 0.009,0.793 ± 0.004,0.000,0.000,651.568 ± 6.184,0.043 ± 0.003,0.795 ± 0.003


In [ ]:
aggregate_mean_std(
    input_file='data/results_ablation_budget.csv',
    output_file='data/results_ablation_budget_aggregated.csv',
    group_cols=ablation_group_cols,
    metric_cols=ablation_metric_cols
)

Processing data/results_ablation_budget.csv...
Found 5 runs per configuration. Calculating Mean ± Std...

Saved aggregated results to data/results_ablation_budget_aggregated.csv



,Dataset,Defense,Queries,Budget_Cap,Surrogate_Acc,Surrogate_Fid,Downstream_Acc,Budget_Consumed,Tau,Latency_ms,KL_Divergence,Attacker_Downstream_Acc
0,ACM,DP-ADAGE,10,1.0,0.117 ± 0.066,0.124 ± 0.078,0.870 ± 0.018,1.000,0.114 ± 0.016,311.761 ± 8.881,6.537 ± 0.480,0.141 ± 0.020
1,ACM,DP-ADAGE,10,10.0,0.140 ± 0.048,0.140 ± 0.052,0.871 ± 0.017,10.000,0.109 ± 0.007,313.008 ± 2.742,6.390 ± 0.266,0.145 ± 0.008
2,ACM,DP-ADAGE,10,200.0,0.460 ± 0.074,0.486 ± 0.084,0.879 ± 0.021,99.059 ± 0.082,0.109 ± 0.007,308.290 ± 5.654,5.330 ± 0.126,0.137 ± 0.024
3,ACM,DP-ADAGE,20,1.0,0.068 ± 0.046,0.074 ± 0.048,0.870 ± 0.018,1.000,0.200 ± 0.029,311.761 ± 8.881,6.537 ± 0.480,0.141 ± 0.020
4,ACM,DP-ADAGE,20,10.0,0.121 ± 0.051,0.125 ± 0.048,0.871 ± 0.017,10.000,0.190 ± 0.022,313.008 ± 2.742,6.390 ± 0.266,0.145 ± 0.008
...,...,...,...,...,...,...,...,...,...,...,...,...
295,Pubmed,DP-ADAGE,190,10.0,0.328 ± 0.040,0.339 ± 0.058,0.791 ± 0.002,10.000,0.957 ± 0.031,875.378 ± 13.145,6.486 ± 2.624,0.333 ± 0.008
296,Pubmed,DP-ADAGE,190,200.0,0.474 ± 0.066,0.492 ± 0.079,0.792 ± 0.003,153.227 ± 17.075,0.946 ± 0.027,884.507 ± 9.304,3.654 ± 0.645,0.324 ± 0.006
297,Pubmed,DP-ADAGE,200,1.0,0.360 ± 0.055,0.369 ± 0.065,0.794 ± 0.003,1.000,0.957 ± 0.024,886.469 ± 22.295,5.536 ± 1.022,0.337 ± 0.009
298,Pubmed,DP-ADAGE,200,10.0,0.325 ± 0.035,0.331 ± 0.060,0.791 ± 0.002,10.000,0.962 ± 0.036,875.378 ± 13.145,6.486 ± 2.624,0.333 ± 0.008


## 5. Total-Budget (ε-Total) Sweep

Additive extension: sweeps `epsilon_total ∈ {1.0, 5.0, 10.0, 200.0}` to evaluate how
the Hard Floor mechanism, noise separation, and benign utility change under different
session budget caps. The existing ε=200 experiment is **not modified**.

### Protocols evaluated
- **`fresh`** — non-binding ledger (paper-compatible benign)
- **`session`** — one cumulative ledger, Hard Floor enforced

### Policies evaluated
- **`benign`** — localized queries (τ ≤ τ_dead), charged ε_max per query
- **`uncertainty`** — attacker with margin-sampling (τ rising gradually)
- **`budget_aware`** — attacker cycling communities

### Key constants
α = 8, δ = 1×10⁻⁶, ε_max = 10.0, ε_0 = 0.01, τ_dead = 0.05

> **Note:** The mock backend (`--backend mock`) runs without a trained GNN and labels
> all output artifacts `[SIMULATED]`. For real results, use `--backend real` after
> completing the environment setup in Sections 1–2.

In [ ]:
# ==============================================================================
# Step 0 (Sweep): Install additional test dependency
# ==============================================================================
!pip install pytest scipy -q

In [ ]:
# ==============================================================================
# Step 1 (Sweep): Run new validation tests
# Validates: cap invariance, tight-cap clamp, floor behavior, sigma limits,
# certificate values, 100% availability, and regression guard for ε=200.
# These tests do NOT touch any existing tests.
# ==============================================================================
!pytest tests/test_sweep_validation.py -v

In [ ]:
# ==============================================================================
# Step 2 (Sweep): Pilot run — fast sanity check
# Caps: {1.0, 200.0}, Dataset: Cora, Seeds: {42, 43}, T=50 queries
# Backend: mock (no trained GNN needed; output labeled [SIMULATED])
# Results → results/sweep_epsilon/
# ==============================================================================
!python run_epsilon_sweep.py --pilot

In [ ]:
# ==============================================================================
# Step 3 (Sweep): Full pilot sweep — all caps, Cora + Pubmed, all 5 seeds
# T=200 for benign; T=500 for attacker policies.
# --resume skips already-completed runs (checkpoint/resume support).
# ==============================================================================
!python run_epsilon_sweep.py \
    --caps 1.0 5.0 10.0 200.0 \
    --datasets cora pubmed \
    --seeds 42 43 44 45 46 \
    --T 200 --T_attacker 500 \
    --resume

python3: can't open file '/content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/run_epsilon_sweep.py': [Errno 2] No such file or directory


In [ ]:
# ==============================================================================
# Step 4 (Sweep): Generate figures and summary from pilot results
# Produces 5 PDFs in reports/figures/ + summary_table.csv
# --audit prints the tau/eps_q trace investigation per the paper's discrepancy
# ==============================================================================
!python summarize_epsilon_sweep.py --audit

In [ ]:
# ==============================================================================
# Step 5 (Sweep): Full 5-dataset grid (long-running)
# Datasets: Cora, Pubmed, Citeseer, ACM (Cora_ML fallback), DBLP
# Total: 4 caps × 5 seeds × 5 datasets × 2 protocols × 3 policies × 2 output_types
#      = 1200 runs (with mock backend, fast; with real GNN, may take hours)
# ==============================================================================
!python run_epsilon_sweep.py \
    --caps 1.0 5.0 10.0 200.0 \
    --datasets cora pubmed citeseer acm dblp \
    --seeds 42 43 44 45 46 \
    --T 200 --T_attacker 500 \
    --resume

In [ ]:
# ==============================================================================
# Step 6 (Sweep): Final summary with all 5 datasets
# ==============================================================================
!python summarize_epsilon_sweep.py

In [ ]:
# ==============================================================================
# Step 7 (Sweep): Optional ablation — static per-query ε vs adaptive allocator
# Runs at each cap to isolate the allocator's contribution.
# ==============================================================================
!python run_epsilon_sweep.py \
    --caps 1.0 5.0 10.0 200.0 \
    --datasets cora pubmed \
    --seeds 42 43 44 45 46 \
    --ablation --resume

python3: can't open file '/content/DP-ADAGE-Differentially-Private-Active-Defenses-Against-GNN-Extraction/run_epsilon_sweep.py': [Errno 2] No such file or directory


In [29]:
# ==============================================================================
# Step 8 (Sweep): Regression guard — verify existing ε=200 experiment unchanged
# Accounting must match to 1e-9; metrics within declared seed tolerance.
# ==============================================================================
!python experiments/exp_main_results.py

Running Main Experiment (Table 1 & Figure 2)...

--- Running with seed 42 ---
Successfully loaded real Cora dataset.
Dataset stats: {'nodes': 2708, 'edges': 10556, 'features': 1433}
Training victim model (GCN)...
Setting up defenses...

Evaluating against NoDefense...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against StaticNoise...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0003

Evaluating against OriginalADAGE...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0001

Evaluating against PRADA...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against DAWN...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0002

Evaluating against DP-ADAGE...
    [Debug] Surrogate Training - Queries: 100, Loss: 0.0468

--- Running with seed 43 ---
Successfully loaded real Cora dataset.
Dataset stats: {'nodes': 2708, 'edges': 10556, 'features': 1433}
Training victim model (GCN)...
Setting up defenses...

Evaluating against No

In [35]:
!zip -r data.zip "data/"

  adding: data/ (stored 0%)
  adding: data/results_adaptive_attacker.csv (deflated 86%)
  adding: data/results_all_datasets_aggregated.csv (deflated 88%)
  adding: data/results_ablation_budget.csv (deflated 85%)
  adding: data/config_default.json (deflated 54%)
  adding: data/results_profiling.csv (stored 0%)
  adding: data/results_main_cora.csv (deflated 77%)
  adding: data/results_all_datasets.csv (deflated 87%)
  adding: data/results_ablation_budget_aggregated.csv (deflated 86%)
  adding: data/p_values_table1.csv (deflated 20%)
  adding: data/results_large_scale.csv (deflated 83%)


In [36]:
!zip -r figurs.zip "figures/"

  adding: figures/ (stored 0%)
  adding: figures/fig2_surrogate_trajectory.pdf (deflated 17%)
  adding: figures/fig4_ablation.pdf (deflated 12%)
  adding: figures/main_results.pdf (deflated 10%)
  adding: figures/fig3_budget_trajectory.pdf (deflated 14%)
  adding: figures/attacker_accuracy_bars.pdf (deflated 9%)
  adding: figures/fig5_benign_vs_attacker.pdf (deflated 9%)
  adding: figures/fig6_adaptive_attacker.pdf (deflated 9%)
  adding: figures/ablation_results.pdf (deflated 11%)


In [37]:
!zip -r results.zip "results/"

  adding: results/ (stored 0%)
  adding: results/sweep_epsilon/ (stored 0%)
  adding: results/sweep_epsilon/summary_table.csv (deflated 90%)


In [38]:
!zip -r reports.zip "reports/"

  adding: reports/ (stored 0%)
  adding: reports/figures/ (stored 0%)
  adding: reports/figures/utility_retention_vs_certificate.pdf (deflated 36%)
  adding: reports/figures/noise_separation_by_cap.pdf (deflated 34%)
  adding: reports/figures/attacker_vs_benign_accuracy.pdf (deflated 28%)
  adding: reports/figures/budget_exhaustion_heatmap.pdf (deflated 32%)
  adding: reports/figures/legitimate_user_session_accuracy.pdf (deflated 25%)
